# E0 Custom CNN from scratch

Nguyễn Anh Kiệt — branch `kiet`. Notebook chỉ chuẩn bị môi trường, kiểm thử và gọi Shared Training Pipeline. Kiến trúc nằm trong `src/models/custom_cnn.py`; không viết training loop tại đây.

Chỉ dùng Train 2.400 ảnh và Validation 300 ảnh. Không đọc Calibration hoặc Official Test. Giữ nguyên config, preprocessing, seed 42, effective batch size 32, max epochs 40 và early stopping 7.

Trước training chính thức: push source lên `kiet` và mở Draft PR về `main` theo Milestone 1. Notebook mặc định chỉ kiểm thử; bật training sau khi hoàn tất mốc này. Không merge PR.


In [ ]:
from pathlib import Path
import hashlib
import io
import json
import os
import platform
import shutil
import subprocess
import sys
import time
import zipfile

REPOSITORY_URL = "https://github.com/DGiang1126/dogbreedid-cv-project.git"
BRANCH = "kiet"
# Có thể pin SHA đã push bằng biến môi trường DOGBREEDID_GIT_COMMIT.
SOURCE_COMMIT = os.environ.get("DOGBREEDID_GIT_COMMIT", "").strip()
RUN_TRAINING = os.environ.get("DOGBREEDID_RUN_TRAINING", "0") == "1"
BATCH_SIZE = 8
NUM_WORKERS = int(os.environ.get("DOGBREEDID_NUM_WORKERS", "0"))
assert NUM_WORKERS >= 0
WORKSPACE = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
PROJECT_ROOT = WORKSPACE / "dogbreedid-cv-project"
if not PROJECT_ROOT.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
if SOURCE_COMMIT:
    assert len(SOURCE_COMMIT) == 40 and all(c in "0123456789abcdefABCDEF" for c in SOURCE_COMMIT), "DOGBREEDID_GIT_COMMIT cần SHA đầy đủ 40 ký tự."
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=PROJECT_ROOT, check=True)
    resolved_commit = subprocess.check_output(["git", "rev-parse", "--verify", SOURCE_COMMIT + "^{commit}"], cwd=PROJECT_ROOT, text=True).strip()
    subprocess.run(["git", "merge-base", "--is-ancestor", resolved_commit, "origin/" + BRANCH], cwd=PROJECT_ROOT, check=True)
    assert not subprocess.check_output(["git", "status", "--porcelain", "--untracked-files=no"], cwd=PROJECT_ROOT, text=True).strip(), "Working copy có thay đổi chưa commit; dừng trước checkout source."
    # Chạy lại source đã được publish trên kiet mà không di chuyển branch kiet.
    subprocess.run(["git", "checkout", "--detach", resolved_commit], cwd=PROJECT_ROOT, check=True)
    assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip() == resolved_commit
else:
    assert subprocess.check_output(["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True).strip() == BRANCH
assert not subprocess.check_output(["git", "diff", "HEAD", "--", "src", "configs", "data/splits", "data/metadata"], cwd=PROJECT_ROOT, text=True).strip(), "Source/config phải sạch và đã commit trước khi chạy."
GIT_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Branch:", BRANCH, "Commit:", GIT_COMMIT)
print("Training enabled:", RUN_TRAINING)


## Môi trường và config

Cài dependencies của repo nếu thiếu; không tự đổi config. Dataset Kaggle dùng bản Stanford Dogs giữ nguyên folder và filename, có thể gắn `jessicali9530/stanford-dogs-dataset`. Nếu chạy ngoài Kaggle, đặt `DOGBREEDID_DATA_ROOT` đến thư mục `stanford_dogs` chứa `Images/`.

Để chạy đúng source của một lượt cũ sau khi nhánh `kiet` có commit mới, đặt `DOGBREEDID_GIT_COMMIT` bằng SHA đầy đủ 40 ký tự trong `run_metadata.json`. Notebook xác minh SHA đã được publish trên `origin/kiet`, rồi checkout source đó ở trạng thái detached HEAD trong working copy của notebook; branch `kiet` không bị di chuyển. Dùng working copy sạch. Mặc định `DOGBREEDID_RUN_TRAINING=0` chỉ chạy prechecks.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
import numpy as np
import pandas as pd
import torch
from PIL import Image
from src.models.model_factory import build_model, build_optimizer
from src.utils.config import load_experiment_config
from src.utils.seed import set_seed

CONFIG_PATH = "configs/experiments/e0_custom_cnn.yaml"
CONFIG = load_experiment_config(CONFIG_PATH)
assert CONFIG["experiment"]["id"] == "E0"
assert CONFIG["experiment"]["seed"] == 42
assert CONFIG["training"]["effective_batch_size"] == 32
assert CONFIG["training"]["max_epochs"] == 40
assert CONFIG["training"]["early_stopping_patience"] == 7
assert CONFIG["training"]["learning_rate"] == 0.001
print("Python:", platform.python_version(), "PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
EXPECTED_SHA256 = {
    "configs/base.yaml": "1f70efd233bc607c99a4947740fb29e52e5324c4d718f859c0d8c4a69cc34709",
    "configs/data.yaml": "add251840c7c6ca71f9d7e99bc9510b232b90afd2842ec0984535ade590d8bb7",
    "configs/experiments/e0_custom_cnn.yaml": "e27ea47489b3f4369408ad0bc5f58023eacc416c45a2fb73bb3e1cf734f41c0f",
    "data/metadata/class_mapping.json": "d86b3a3a232babea289070299727d20d56b8f1607593ce1aaf8b174465690c3c",
    "data/splits/train.csv": "6328f08544e33049e1feccd1f65892c517d727ddb27ba941d6a690fafe1e6aba",
    "data/splits/validation.csv": "3f6dbe00e9c120d8c7acabcaa658830011beb1ea989949a07471542ac95c8d13"
}
for name, expected in EXPECTED_SHA256.items():
    assert hashlib.sha256((PROJECT_ROOT / name).read_bytes()).hexdigest() == expected, f"Frozen file changed: {name}"
print("Frozen config, mapping, Train/Validation split hashes verified.")


## Dataset và kiểm thử trước training

Chỉ kiểm tra ảnh được liệt kê trong `train.csv` và `validation.csv`. Không chia lại dữ liệu, không đổi nhãn. Dry-run không thay thế Model Smoke Test.


In [ ]:
DATA_ROOT_SETTING = os.environ.get("DOGBREEDID_DATA_ROOT", "").strip()
if DATA_ROOT_SETTING:
    DATA_ROOT = Path(DATA_ROOT_SETTING).expanduser().resolve()
else:
    candidates = []
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        candidates = sorted(path for path in input_root.rglob("Images") if path.is_dir())
    default_root = PROJECT_ROOT / "data/raw/stanford_dogs"
    if (default_root / "Images").is_dir():
        candidates.append(default_root / "Images")
    frames = {name: pd.read_csv(PROJECT_ROOT / f"data/splits/{name}.csv") for name in ("train", "validation")}
    sample_paths = [frame.iloc[0]["relative_path"] for frame in frames.values()]
    candidates = sorted({path.resolve() for path in candidates if all((path / relative).is_file() for relative in sample_paths)})
    assert len(candidates) == 1, "Gắn dataset Stanford Dogs hoặc đặt DOGBREEDID_DATA_ROOT đến thư mục chứa Images/."
    # Wrapper path stanford_dogs/Images đúng quy ước; symlink không sửa ảnh gốc.
    DATA_ROOT = default_root
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    image_link = DATA_ROOT / "Images"
    if not image_link.exists():
        image_link.symlink_to(candidates[0], target_is_directory=True)
assert (DATA_ROOT / "Images").is_dir()
frames = {name: pd.read_csv(PROJECT_ROOT / f"data/splits/{name}.csv") for name in ("train", "validation")}
assert len(frames["train"]) == 2400 and len(frames["validation"]) == 300
assert set(frames["train"].relative_path).isdisjoint(frames["validation"].relative_path)
for name, frame in frames.items():
    assert frame.class_id.nunique() == 30
    assert frame.groupby("class_id").size().eq(80 if name == "train" else 10).all()
    for relative in frame.relative_path:
        path = DATA_ROOT / "Images" / relative
        assert path.is_file(), f"Missing {name} image: {relative}"
        with Image.open(path) as image:
            image.verify()
print("Train=2400, Validation=300; 30 lớp; ảnh tồn tại/đọc được; không overlap.")


In [ ]:
set_seed(42)
model = build_model(CONFIG)
assert isinstance(model, torch.nn.Module)
assert all(parameter.requires_grad for parameter in model.parameters())
assert not any(isinstance(module, (torch.nn.Softmax, torch.nn.LogSoftmax)) for module in model.modules())
assert sum(isinstance(module, torch.nn.Conv2d) for module in model.modules()) == 4
model.eval()
with torch.inference_mode():
    logits = model(torch.randn(2, 3, 224, 224))
assert logits.shape == (2, 30) and torch.isfinite(logits).all()
optimizer = build_optimizer(model, CONFIG)
assert isinstance(optimizer, torch.optim.AdamW)
assert len(optimizer.param_groups) == 1
assert optimizer.param_groups[0]["lr"] == 1e-3
assert optimizer.param_groups[0]["weight_decay"] == 1e-4
assert {id(p) for p in model.parameters()} == {id(p) for p in optimizer.param_groups[0]["params"]}
# Kiểm tra backward/update với dữ liệu giả; không lưu thành kết quả thực nghiệm.
model.train()
before = model.classifier[-1].weight.detach().clone()
optimizer.zero_grad(set_to_none=True)
loss = torch.nn.CrossEntropyLoss()(model(torch.randn(2, 3, 224, 224)), torch.tensor([0, 1]))
loss.backward()
assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters())
optimizer.step()
assert not torch.equal(before, model.classifier[-1].weight)
PARAMETER_COUNT = sum(p.numel() for p in model.parameters())
print("MODEL SMOKE TEST PASSED; raw logits [2,30]; all trainable; parameters:", PARAMETER_COUNT)
del model, optimizer


In [ ]:
LOG_DIR = PROJECT_ROOT / "outputs/logs/E0/seed_42"
LOG_DIR.mkdir(parents=True, exist_ok=True)
TEST_COMMAND = [sys.executable, "-m", "pytest", "-q"]
tests = subprocess.run(TEST_COMMAND, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(tests.stdout)
assert tests.returncode == 0, "Unit tests failed; do not start training."
COMMON_COMMAND = [sys.executable, "-u", "scripts/train.py", "--config", CONFIG_PATH,
                  "--dataset-root", str(DATA_ROOT), "--batch-size", str(BATCH_SIZE),
                  "--num-workers", str(NUM_WORKERS), "--device", "auto"]
dry_run = subprocess.run(COMMON_COMMAND + ["--dry-run"], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(dry_run.stdout)
assert dry_run.returncode == 0 and "DATA DRY-RUN PASSED" in dry_run.stdout
PRECHECKS = {"git_commit": GIT_COMMIT, "model_smoke_test": "passed", "output_shape": [2, 30],
            "total_parameters": PARAMETER_COUNT, "trainable_parameters": PARAMETER_COUNT,
            "unit_tests": tests.stdout, "dataloader_dry_run": dry_run.stdout,
            "frozen_sha256": EXPECTED_SHA256}


## Training chính thức và bàn giao

Sau khi source đã push và Draft PR đã mở, đặt biến môi trường `DOGBREEDID_RUN_TRAINING=1` trước khi chạy notebook. Chạy Shared Runner nguyên bản, không giảm epochs hoặc thay siêu tham số. CUDA được yêu cầu cho lượt chạy chính thức trên Kaggle.

Checkpoint `.pt` lưu trong output của phiên Kaggle và backup vào nơi lâu dài có thể chia sẻ; không commit vào GitHub. Chỉ commit notebook, source, output CSV/JSON và Task Report đúng thư mục của Kiệt. Không ghi đè kết quả E0 đã tồn tại.


In [ ]:
if RUN_TRAINING:
    assert torch.cuda.is_available(), "Không có GPU; dừng trước official training."
    expected_paths = [PROJECT_ROOT / f"outputs/checkpoints/E0/seed_42/{name}" for name in ("best.pt", "last.pt")]
    expected_paths += [LOG_DIR / "history.csv", LOG_DIR / "run_metadata.json",
                       PROJECT_ROOT / "outputs/metrics/E0/seed_42/validation.json"]
    assert not any(path.exists() for path in expected_paths), "Không ghi đè kết quả đã hoàn thành."
    start = time.perf_counter()
    started_at = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    subprocess.run(COMMON_COMMAND, check=True)
    duration = time.perf_counter() - start
    assert all(path.is_file() for path in expected_paths), "Thiếu output bắt buộc."
    summary = {**PRECHECKS, "started_at_utc": started_at, "training_duration_seconds": duration,
               "training_command": COMMON_COMMAND, "python_version": platform.python_version(),
               "torch_version": torch.__version__, "cuda_version": torch.version.cuda,
               "gpu": torch.cuda.get_device_name(0), "physical_batch_size": BATCH_SIZE,
               "effective_batch_size": 32, "seed": 42}
    (LOG_DIR / "execution_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
    metrics_path = PROJECT_ROOT / "outputs/metrics/E0/seed_42/validation.json"
    print("Best validation metrics:", metrics_path.read_text())
    print("Training duration (seconds):", duration)
    # Backup đúng năm output bắt buộc cùng evidence trước khi phiên Kaggle kết thúc.
    archive_path = WORKSPACE / "E0_seed_42_artifacts.zip"
    assert not archive_path.exists(), "Không ghi đè archive kết quả."
    with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in expected_paths + [LOG_DIR / "execution_summary.json"]:
            archive.write(path, path.relative_to(PROJECT_ROOT))
    print("Backup:", archive_path)
else:
    print("Prechecks passed. Chưa train: cần push source, mở Draft PR và bật DOGBREEDID_RUN_TRAINING=1.")
